# 12c · Sensibilidad de κ con reentrenamiento completo

Este cuaderno determina si las conclusiones predictivas, estructurales y
biológicas dependen del objetivo de sparsidad utilizado durante el
aprendizaje. Ejecuta el flujo completo —Phase 1, auditoría de soporte y
Phase 2— para cinco densidades:

\[
\kappa\in\{0.005,0.007,0.010,0.015,0.020\}.
\]

El entrenamiento principal \(\kappa=0.010\) se valida y reutiliza. Por
tanto, con los artefactos principales completos se realizan veinte
entrenamientos nuevos: cuatro densidades × cinco semillas.

Reglas del análisis:

- se modifica exclusivamente `KEEP_MIN`;
- se conservan semillas, splits, presupuesto, pérdidas y checkpoints;
- cada densidad usa artefactos, contratos y fingerprints independientes;
- toda evaluación procede de desarrollo-validación;
- no se abre ni se usa ninguna cohorte externa;
- κ=0.010 permanece como especificación principal: el barrido no se usa
  para escoger retrospectivamente el resultado más favorable.

**No ejecute este notebook simultáneamente con las permutaciones del
notebook 12.** Ambos entrenamientos compiten por GPU, CPU y E/S. Espere a
que las permutaciones finalicen y después ejecute `Run All`. El cuaderno
es reanudable y guarda progreso después de cada semilla.


## 1. Imports, raíz del proyecto y protocolo principal


In [ ]:
from __future__ import annotations

from contextlib import contextmanager, nullcontext
from dataclasses import asdict
from functools import lru_cache
from hashlib import sha256
from itertools import combinations
from pathlib import Path
from textwrap import dedent
import inspect
import json
import math
import os
import sys
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/tmp/intergate-mpl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import t as student_t

try:
    from IPython.display import Markdown, display
except ImportError:
    display = print

    def Markdown(value):
        return value


HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in (HERE, *HERE.parents)
        if (path / "intergate").is_dir()
        and (path / "configs" / "protocol_v8.json").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encuentra la raíz del repositorio. Abra el notebook "
        "desde InterGATE o uno de sus subdirectorios."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.ablation import (
    CFG_FULL,
    build_pruned_model_from_bundle,
    cfg_to_dict,
    load_bundle,
)
from intergate.biology import combine_biology_scores, module_aware_exact_k
from intergate.config import (
    CFG,
    implementation_fingerprint,
    model_config_fingerprint,
    model_config_snapshot,
)
from intergate.data import ExpressionDataset
from intergate.losses import compute_metrics_full
from intergate.posthoc_biology_common import (
    load_locked_signature_resources,
    safe_name,
)
from intergate.revision_analyses import aggregate_seed_graphs, topology_metrics
from intergate.runtime_setup import (
    assert_transport_protocol,
    assert_v8_protocol,
    prepare_runtime,
)
from intergate.training import predict_proba_xh_mode
from intergate.transport import TransportConfig, run_transport_seed
from intergate.utils import cleanup_memory


plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})


In [ ]:
def sha256_file(path):
    digest = sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_json_sha256(payload):
    data = json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
    return sha256(data).hexdigest()


def first_existing(values, *, predicate=lambda path: path.exists(), label="ruta"):
    candidates = [
        Path(value).expanduser().resolve()
        for value in values
        if value is not None and str(value).strip()
    ]
    for path in candidates:
        if predicate(path):
            return path
    raise FileNotFoundError(
        f"No se encontró {label}. Candidatas:\n- "
        + "\n- ".join(map(str, candidates))
    )


PROTOCOL_FILE = PROJECT_ROOT / "configs" / "protocol_v8.json"
PROTOCOL = json.loads(PROTOCOL_FILE.read_text(encoding="utf-8"))
KAPPA_GRID = (0.005, 0.007, 0.010, 0.015, 0.020)
PRIMARY_KAPPA = 0.010
SEEDS = tuple(map(int, PROTOCOL["outer_seeds"]))
CLASS_NAMES = tuple(map(str, PROTOCOL["endpoint"]))
TARGET_CLASS = "Basal-like"
NONINFERIORITY_MARGIN = 0.010

if tuple(SEEDS) != tuple(map(int, CFG.STABILITY_SEEDS)):
    raise RuntimeError("Las semillas del protocolo y CFG no coinciden.")
if TARGET_CLASS not in CLASS_NAMES:
    raise RuntimeError(f"{TARGET_CLASS!r} no pertenece al endpoint.")
if not math.isclose(float(CFG.KEEP_MIN), PRIMARY_KAPPA, abs_tol=1e-12):
    raise RuntimeError(
        f"El kernel debe comenzar con KEEP_MIN={PRIMARY_KAPPA}; "
        f"observado={CFG.KEEP_MIN}. Reinicie el kernel."
    )

# La validación principal se realiza antes de modificar κ. El barrido
# posterior usa un contrato específico que permite cambiar solo KEEP_MIN.
assert_v8_protocol(CFG)
assert_transport_protocol(CFG)

RESULTS_ROOT = Path(
    os.environ.get("INTERGATE_RESULTS_ROOT", PROJECT_ROOT / "results_v8")
).expanduser().resolve()
ARTIFACTS_ROOT = Path(CFG.ARTIFACTS_ROOT).expanduser().resolve()
OUTPUT_DIR = RESULTS_ROOT / "12c_kappa_retraining"
CONTRACT_DIR = OUTPUT_DIR / "contracts"
GRAPH_DIR = OUTPUT_DIR / "graphs"
FIGURE_DIR = OUTPUT_DIR / "figures"
for path in (OUTPUT_DIR, CONTRACT_DIR, GRAPH_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

PRIMARY_ARTIFACT_NAME = str(
    getattr(CFG, "V8_ARTIFACT_NAME", "INTERGATE_V8")
)


def kappa_tag(kappa):
    return f"{float(kappa):.3f}".replace(".", "p")


ARTIFACT_BY_KAPPA = {
    float(kappa): (
        PRIMARY_ARTIFACT_NAME
        if math.isclose(kappa, PRIMARY_KAPPA, abs_tol=1e-12)
        else f"INTERGATE_KSENS_{kappa_tag(kappa)}"
    )
    for kappa in KAPPA_GRID
}

RUN_TRAINING = os.environ.get("INTERGATE_RUN_KAPPA_TRAINING", "1") == "1"
RUN_BRANCH_AUDIT = os.environ.get("INTERGATE_RUN_KAPPA_BRANCH_AUDIT", "1") == "1"
RESUME = True
FORCE = False

PHASE1_SAMPLES_PER_FLOW = (
    int(CFG.GRAPH_BATCH_SIZE) * int(CFG.TRANSPORT_STEPS_PER_EPISODE)
)
PHASE1_OPTIMIZER_STEPS = (
    int(CFG.TRANSPORT_STEPS_PER_EPISODE)
    + int(CFG.GRAPH_ACCUM_STEPS) - 1
) // int(CFG.GRAPH_ACCUM_STEPS)
SUPPORT_SAMPLES_PER_STEP = (
    int(CFG.GRAPH_BATCH_SIZE) * int(CFG.TRANSPORT_SUPPORT_BATCHES)
)
if (
    PHASE1_SAMPLES_PER_FLOW,
    PHASE1_OPTIMIZER_STEPS,
    SUPPORT_SAMPLES_PER_STEP,
) != (384, 2, 128):
    raise RuntimeError("El presupuesto de entrenamiento no es el pareado.")

configuration = pd.Series({
    "project_root": str(PROJECT_ROOT),
    "results_root": str(RESULTS_ROOT),
    "artifacts_root": str(ARTIFACTS_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "kappa_grid": KAPPA_GRID,
    "primary_kappa": PRIMARY_KAPPA,
    "seeds": SEEDS,
    "new_complete_runs": (len(KAPPA_GRID) - 1) * len(SEEDS),
    "run_training": RUN_TRAINING,
    "run_branch_audit": RUN_BRANCH_AUDIT,
    "resume": RESUME,
    "force": FORCE,
    "external_opened": False,
}, name="value")
display(configuration.to_frame())
display(pd.DataFrame({
    "kappa": KAPPA_GRID,
    "artifact_name": [ARTIFACT_BY_KAPPA[k] for k in KAPPA_GRID],
    "role": [
        "primary_reused" if k == PRIMARY_KAPPA else "sensitivity_retraining"
        for k in KAPPA_GRID
    ],
}))


## 2. Runtime de desarrollo y aislamiento de la intervención

El runtime se construye una sola vez con la configuración principal
validada. Después se comprueba que cada configuración de sensibilidad
difiere de ella únicamente en `KEEP_MIN`. Los datos externos no se leen.


In [ ]:
runtime = prepare_runtime(CFG, include_external=False, validate=True)
if bool(runtime["external_opened"]) or len(runtime["test_idx"]):
    raise RuntimeError("El runtime de sensibilidad ha abierto el externo.")
if tuple(map(str, runtime["classes"])) != CLASS_NAMES:
    raise RuntimeError("El orden de clases no coincide con el protocolo.")

BASE_SNAPSHOT = model_config_snapshot(CFG)
BASE_FINGERPRINT = model_config_fingerprint(CFG)
IMPLEMENTATION_SHA256 = implementation_fingerprint()
N_CANDIDATE_EDGES = int(runtime["candidate_edges"])
K_TABLE = pd.DataFrame({"kappa": KAPPA_GRID})
K_TABLE["K"] = np.rint(K_TABLE.kappa * N_CANDIDATE_EDGES).astype(int)
K_BY_KAPPA = dict(zip(K_TABLE.kappa.astype(float), K_TABLE.K.astype(int)))

if K_TABLE.K.duplicated().any():
    raise RuntimeError("Dos valores de κ producen el mismo K.")
if not math.isclose(float(runtime["kappa"]), PRIMARY_KAPPA, abs_tol=1e-12):
    raise RuntimeError("El runtime principal no tiene κ=0.010.")


def snapshot_differences(reference, current):
    keys = sorted(set(reference) | set(current))
    return {
        key: {"primary": reference.get(key), "sensitivity": current.get(key)}
        for key in keys
        if reference.get(key) != current.get(key)
    }


configuration_audit_rows = []
original_kappa = float(CFG.KEEP_MIN)
try:
    for kappa in KAPPA_GRID:
        CFG.KEEP_MIN = float(kappa)
        current_snapshot = model_config_snapshot(CFG)
        differences = snapshot_differences(BASE_SNAPSHOT, current_snapshot)
        expected = set() if kappa == PRIMARY_KAPPA else {"KEEP_MIN"}
        if set(differences) != expected:
            raise RuntimeError(
                f"κ={kappa}: cambiaron campos no autorizados: {differences}."
            )
        configuration_audit_rows.append({
            "kappa": kappa,
            "K": K_BY_KAPPA[kappa],
            "artifact_name": ARTIFACT_BY_KAPPA[kappa],
            "changed_fields": ",".join(sorted(differences)) or "none_primary",
            "model_config_sha256": model_config_fingerprint(CFG),
            "implementation_sha256": IMPLEMENTATION_SHA256,
        })
finally:
    CFG.KEEP_MIN = original_kappa

configuration_audit = pd.DataFrame(configuration_audit_rows)
if configuration_audit.model_config_sha256.nunique() != len(KAPPA_GRID):
    raise RuntimeError("Los cinco valores de κ no generan fingerprints distintos.")
configuration_audit.to_csv(
    OUTPUT_DIR / "kappa_configuration_isolation_audit.csv", index=False
)
display(K_TABLE)
display(configuration_audit)
print(
    f"Desarrollo: train={len(runtime['train_idx'])}, "
    f"val={len(runtime['val_idx'])}, test={len(runtime['test_idx'])}; "
    f"prior elegible={N_CANDIDATE_EDGES:,}."
)


## 3. Contratos y entrenamiento reanudable

El orden de ejecución valida primero el modelo principal y después
entrena las cuatro densidades alternativas. Cada semilla produce el
mismo conjunto de artefactos que el notebook 02. Si el kernel se
interrumpe, vuelva a ejecutar el cuaderno: `force=False` reanuda el
checkpoint compatible y reutiliza los bundles completos.


In [ ]:
REQUIRED_ARTIFACT_FILES = (
    "meta.json",
    "graph_bundle.pt",
    "biology_graph_bundle.pt",
    "model_pruned.pt",
    "transport_support_dense.npz",
    "transport_support_summary.json",
    "transport_adaptation_audit.csv",
)


def sensitivity_contract(kappa):
    current_snapshot = model_config_snapshot(CFG)
    differences = snapshot_differences(BASE_SNAPSHOT, current_snapshot)
    return {
        "analysis": "kappa_training_target_sensitivity",
        "kappa": float(kappa),
        "K_nominal": int(K_BY_KAPPA[float(kappa)]),
        "candidate_edges": N_CANDIDATE_EDGES,
        "artifact_name": ARTIFACT_BY_KAPPA[float(kappa)],
        "seeds": list(map(int, SEEDS)),
        "primary_kappa": PRIMARY_KAPPA,
        "changed_fields": differences,
        "model_config_sha256": model_config_fingerprint(CFG),
        "implementation_sha256": IMPLEMENTATION_SHA256,
        "base_model_config_sha256": BASE_FINGERPRINT,
        "external_opened": False,
        "evaluation_split": "development_validation",
        "phase1_samples_per_flow": PHASE1_SAMPLES_PER_FLOW,
        "phase1_optimizer_steps_per_episode": PHASE1_OPTIMIZER_STEPS,
        "support_samples_per_step": SUPPORT_SAMPLES_PER_STEP,
    }


def write_or_validate_contract(kappa, artifact_root):
    contract = sensitivity_contract(kappa)
    result_contract = CONTRACT_DIR / f"contract_{kappa_tag(kappa)}.json"
    if result_contract.is_file():
        observed = json.loads(result_contract.read_text(encoding="utf-8"))
        if observed != contract:
            raise RuntimeError(f"Contrato incompatible: {result_contract}")
    else:
        result_contract.write_text(
            json.dumps(contract, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

    # El modelo principal es un artefacto previo e inmutable. Los
    # contratos internos se añaden solo a los nuevos brazos.
    if not math.isclose(kappa, PRIMARY_KAPPA, abs_tol=1e-12):
        artifact_contract = artifact_root / "KAPPA_SENSITIVITY_CONTRACT.json"
        artifact_root.mkdir(parents=True, exist_ok=True)
        if artifact_contract.is_file():
            observed = json.loads(
                artifact_contract.read_text(encoding="utf-8")
            )
            if observed != contract:
                raise RuntimeError(
                    f"Contrato incompatible: {artifact_contract}"
                )
        else:
            unexpected = [
                path for path in artifact_root.iterdir()
                if path.name != artifact_contract.name
            ]
            if unexpected:
                raise RuntimeError(
                    f"{artifact_root} ya contiene artefactos sin contrato. "
                    "No se reutilizan automáticamente."
                )
            artifact_contract.write_text(
                json.dumps(contract, indent=2, ensure_ascii=False),
                encoding="utf-8",
            )
    return contract


def assert_partial_checkpoint_compatible(run_dir, contract):
    checkpoint = run_dir / "transport_last_checkpoint.pt"
    if not checkpoint.is_file():
        return
    payload = torch.load(
        checkpoint, map_location="cpu", weights_only=False
    )
    if payload.get("model_config_sha256") != contract["model_config_sha256"]:
        raise RuntimeError(
            f"{checkpoint}: fingerprint de configuración incompatible."
        )
    if payload.get("implementation_sha256") != IMPLEMENTATION_SHA256:
        raise RuntimeError(
            f"{checkpoint}: fingerprint de implementación incompatible."
        )


def load_completed_result_or_fail(
    run_dir, *, contract, transport, artifact_name
):
    missing = [
        name for name in REQUIRED_ARTIFACT_FILES
        if not (run_dir / name).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"Artefacto congelado incompleto en {run_dir}: {missing}."
        )
    meta = json.loads(
        (run_dir / "meta.json").read_text(encoding="utf-8")
    )
    protocol = meta.get("transport_protocol", {})
    if protocol.get("completed") is not True:
        raise RuntimeError(f"{run_dir}: el bundle no está completo.")
    if str(meta.get("cfg_name")) != str(artifact_name):
        raise RuntimeError(f"{run_dir}: cfg_name no coincide.")
    bad_transport = {
        key: (protocol.get(key), value)
        for key, value in asdict(transport).items()
        if protocol.get(key) != value
    }
    if bad_transport:
        raise RuntimeError(
            f"{run_dir}: contrato de transporte incompatible: "
            f"{bad_transport}."
        )
    if meta.get("model_config_sha256") != contract["model_config_sha256"]:
        raise RuntimeError(
            f"{run_dir}: fingerprint de configuración incompatible."
        )
    if meta.get("implementation_sha256") != IMPLEMENTATION_SHA256:
        raise RuntimeError(
            f"{run_dir}: fingerprint de implementación incompatible."
        )
    current = cfg_to_dict(CFG_FULL)
    if meta.get("dataset_signature") != current.get("dataset_signature"):
        raise RuntimeError(
            f"{run_dir}: la firma del dataset no coincide con el runtime."
        )
    metrics = dict(meta.get("metrics", {}))
    if bool(metrics.get("external_opened", True)):
        raise RuntimeError(f"{run_dir}: el bundle abrió el externo.")
    return metrics


RUN_ORDER = (PRIMARY_KAPPA, 0.005, 0.007, 0.015, 0.020)
training_rows = []
progress_file = OUTPUT_DIR / "kappa_training_progress.csv"
original_kappa = float(CFG.KEEP_MIN)
try:
    for kappa in RUN_ORDER:
        CFG.KEEP_MIN = float(kappa)
        artifact_name = ARTIFACT_BY_KAPPA[kappa]
        artifact_root = ARTIFACTS_ROOT / artifact_name
        contract = write_or_validate_contract(kappa, artifact_root)
        transport = TransportConfig.from_global(CFG)
        print(
            f"\n========== κ={kappa:.3f} | "
            f"K={K_BY_KAPPA[kappa]} | {artifact_name} ==========",
            flush=True,
        )
        for position, seed in enumerate(SEEDS, 1):
            run_dir = artifact_root / f"seed_{int(seed)}"
            assert_partial_checkpoint_compatible(run_dir, contract)
            if math.isclose(kappa, PRIMARY_KAPPA, abs_tol=1e-12):
                # El brazo principal es inmutable: se verifica y se lee,
                # pero nunca se reentrena ni se sobrescribe desde 12c.
                row = load_completed_result_or_fail(
                    run_dir,
                    contract=contract,
                    transport=transport,
                    artifact_name=artifact_name,
                )
                print(
                    f"[principal congelado] semilla {seed}: reutilizada",
                    flush=True,
                )
            elif RUN_TRAINING:
                print(
                    f"\n--- semilla {position}/{len(SEEDS)}: {seed} ---",
                    flush=True,
                )
                row = run_transport_seed(
                    int(seed),
                    save_root=str(ARTIFACTS_ROOT),
                    transport=transport,
                    resume=RESUME,
                    force=FORCE,
                    artifact_name=artifact_name,
                )
            else:
                row = load_completed_result_or_fail(
                    run_dir,
                    contract=contract,
                    transport=transport,
                    artifact_name=artifact_name,
                )
            record = {
                "kappa": float(kappa),
                "K_nominal": int(K_BY_KAPPA[kappa]),
                "artifact_name": artifact_name,
                **row,
            }
            training_rows = [
                previous for previous in training_rows
                if not (
                    math.isclose(
                        float(previous["kappa"]), kappa, abs_tol=1e-12
                    )
                    and int(previous["seed"]) == int(seed)
                )
            ]
            training_rows.append(record)
            pd.DataFrame(training_rows).sort_values(
                ["kappa", "seed"]
            ).to_csv(progress_file, index=False)
            display(pd.DataFrame([record])[[
                "kappa", "K_nominal", "seed", "macro_f1", "bal_acc",
                "auc_macro_ovr", "class_graph_edges_per_class",
                "val_macro_f1_phase1_transport_consensus",
                "transport_pairwise_jaccard_mean",
            ]])
            if bool(record.get("transport_all_parameter_deltas_zero", False)):
                raise RuntimeError(
                    f"κ={kappa}/seed={seed}: adaptación estructural nula."
                )
finally:
    CFG.KEEP_MIN = original_kappa
    cleanup_memory("after_kappa_training_sweep")

development_results = pd.DataFrame(training_rows).sort_values(
    ["kappa", "seed"]
).reset_index(drop=True)
if len(development_results) != len(KAPPA_GRID) * len(SEEDS):
    raise RuntimeError(
        f"Barrido incompleto: {len(development_results)} filas; "
        f"esperadas={len(KAPPA_GRID) * len(SEEDS)}."
    )
development_results.to_csv(
    OUTPUT_DIR / "kappa_development_results.csv", index=False
)
display(development_results)


## 4. Integridad de los 25 bundles y métricas predictivas pareadas


In [ ]:
integrity_rows = []
for kappa in KAPPA_GRID:
    artifact_name = ARTIFACT_BY_KAPPA[kappa]
    artifact_root = ARTIFACTS_ROOT / artifact_name
    expected_fingerprint = configuration_audit.loc[
        configuration_audit.kappa.eq(kappa), "model_config_sha256"
    ].iloc[0]
    for seed in SEEDS:
        run_dir = artifact_root / f"seed_{int(seed)}"
        missing = [
            name for name in REQUIRED_ARTIFACT_FILES
            if not (run_dir / name).is_file()
        ]
        if missing:
            raise FileNotFoundError(
                f"κ={kappa}/seed={seed}: faltan {missing}."
            )
        meta = json.loads(
            (run_dir / "meta.json").read_text(encoding="utf-8")
        )
        metrics = meta.get("metrics", {})
        observed_kappa = float(metrics["class_graph_keep_ratio"])
        observed_k = int(metrics["class_graph_edges_per_class"])
        if not math.isclose(observed_kappa, kappa, abs_tol=1e-12):
            raise RuntimeError(
                f"κ={kappa}/seed={seed}: keep_ratio={observed_kappa}."
            )
        if observed_k != int(K_BY_KAPPA[kappa]):
            raise RuntimeError(
                f"κ={kappa}/seed={seed}: Exact-K={observed_k}; "
                f"esperado={K_BY_KAPPA[kappa]}."
            )
        if bool(metrics.get("external_opened", True)):
            raise RuntimeError(f"κ={kappa}/seed={seed}: externo abierto.")
        if meta.get("model_config_sha256") != expected_fingerprint:
            raise RuntimeError(
                f"κ={kappa}/seed={seed}: fingerprint incorrecto."
            )
        integrity_rows.append({
            "kappa": kappa,
            "K": observed_k,
            "seed": int(seed),
            "artifact_name": artifact_name,
            "model_config_sha256": meta.get("model_config_sha256"),
            "implementation_sha256": meta.get("implementation_sha256"),
            "external_opened": False,
            "complete": True,
        })

integrity_audit = pd.DataFrame(integrity_rows)
integrity_audit.to_csv(
    OUTPUT_DIR / "kappa_bundle_integrity_audit.csv", index=False
)

metric_columns = ["macro_f1", "bal_acc", "auc_macro_ovr"]
# Agregación nombrada: evita columnas MultiIndex y conserva una tabla
# idéntica entre versiones de pandas.
predictive_summary = (
    development_results.groupby(["kappa", "K_nominal"])
    .agg(
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        bal_acc_mean=("bal_acc", "mean"),
        bal_acc_std=("bal_acc", "std"),
        auc_macro_ovr_mean=("auc_macro_ovr", "mean"),
        auc_macro_ovr_std=("auc_macro_ovr", "std"),
    )
    .reset_index()
    .rename(columns={"K_nominal": "K"})
)

primary_seed = development_results.loc[
    development_results.kappa.eq(PRIMARY_KAPPA),
    ["seed", "macro_f1", "bal_acc", "auc_macro_ovr"],
].set_index("seed")
paired_rows = []
for kappa in KAPPA_GRID:
    block = development_results.loc[
        development_results.kappa.eq(kappa),
        ["seed", "macro_f1", "bal_acc", "auc_macro_ovr"],
    ].set_index("seed").loc[list(SEEDS)]
    for metric in metric_columns:
        delta = block[metric] - primary_seed.loc[list(SEEDS), metric]
        mean_delta = float(delta.mean())
        sd_delta = float(delta.std(ddof=1))
        critical = float(student_t.ppf(0.975, df=len(delta) - 1))
        half_width = critical * sd_delta / math.sqrt(len(delta))
        paired_rows.append({
            "kappa": kappa,
            "K": K_BY_KAPPA[kappa],
            "metric": metric,
            "mean_delta_vs_primary": mean_delta,
            "sd_delta": sd_delta,
            "paired_seed_ci95_lower": mean_delta - half_width,
            "paired_seed_ci95_upper": mean_delta + half_width,
            "noninferiority_margin": (
                -NONINFERIORITY_MARGIN if metric == "macro_f1" else np.nan
            ),
            "macro_f1_noninferior_diagnostic": (
                bool(mean_delta - half_width > -NONINFERIORITY_MARGIN)
                if metric == "macro_f1" else np.nan
            ),
            "n_paired_seeds": len(delta),
        })
predictive_paired = pd.DataFrame(paired_rows)
predictive_summary.to_csv(
    OUTPUT_DIR / "kappa_predictive_summary.csv", index=False
)
predictive_paired.to_csv(
    OUTPUT_DIR / "kappa_predictive_paired_vs_primary.csv", index=False
)
display(predictive_summary)
display(predictive_paired.loc[predictive_paired.metric.eq("macro_f1")])


## 5. Reconstrucción biológica definitiva para cada entrenamiento

Los artefactos densos conservan la puntuación de todo el prior. Para
mantener la definición usada en el manuscrito, cada red se reconstruye
con pesos 1/0.35/0.075 y el mismo selector modular Exact-K. No se usan
firmas conocidas durante la selección: las firmas se incorporan después,
únicamente para validación.


In [ ]:
CANDIDATE_FILE = (
    RESULTS_ROOT / "05_biology_validation" / "candidate_prior_for_null.csv.gz"
)
SIGNATURE_MANIFEST_FILE = (
    RESULTS_ROOT / "05_biology_validation" / "locked_signature_manifest.json"
)
if not CANDIDATE_FILE.is_file() or not SIGNATURE_MANIFEST_FILE.is_file():
    raise FileNotFoundError(
        "Faltan los resultados bloqueados del notebook 05."
    )
candidates = pd.read_csv(CANDIDATE_FILE)
required_candidate_columns = {
    "candidate_edge_id", "source_idx", "target_idx", "source_gene",
    "target_gene", "prior_weight", "edge_type", "sign",
    "source_degree_bin", "target_degree_bin", "null_stratum",
}
missing = required_candidate_columns - set(candidates.columns)
if missing:
    raise ValueError(f"El prior candidato carece de {sorted(missing)}")
candidates = candidates.sort_values("candidate_edge_id").reset_index(drop=True)
candidates["candidate_edge_id"] = candidates.candidate_edge_id.astype(int)
if candidates.candidate_edge_id.duplicated().any():
    raise RuntimeError("candidate_edge_id contiene duplicados.")
for column in ("source_gene", "target_gene"):
    candidates[column] = candidates[column].astype(str).str.upper()
candidates["null_stratum"] = candidates.null_stratum.astype(str)
if len(candidates) != N_CANDIDATE_EDGES:
    raise RuntimeError(
        f"Prior nulo={len(candidates)} aristas; runtime={N_CANDIDATE_EDGES}."
    )

candidate_lookup = candidates.set_index("candidate_edge_id", drop=False)
candidate_ids = candidates.candidate_edge_id.to_numpy(np.int64)
dense_edge_capacity = int(candidate_ids.max()) + 1
candidate_id_to_row = np.full(dense_edge_capacity, -1, dtype=np.int64)
candidate_id_to_row[candidate_ids] = np.arange(len(candidates), dtype=np.int64)
full_edge_index = np.full((2, dense_edge_capacity), -1, dtype=np.int64)
full_edge_index[:, candidate_ids] = candidates[
    ["source_idx", "target_idx"]
].to_numpy(np.int64).T

resource_candidates = [
    value for value in (
        os.environ.get("INTERGATE_BIOLOGY_RESOURCES"),
        PROJECT_ROOT / "resources_biological",
        RESULTS_ROOT.parent / "resources_biological",
    ) if value is not None
]
locked_signature_manifest = json.loads(
    SIGNATURE_MANIFEST_FILE.read_text(encoding="utf-8")
)
locked_msigdb_release = str(
    locked_signature_manifest["msigdb_release"]
)
RESOURCE_DIR = first_existing(
    resource_candidates,
    predicate=lambda path: (
        (path / "PAM50_symbols.txt").is_file()
        and (
            path / "msigdb"
            / f"c2.cgp.v{locked_msigdb_release}.symbols.gmt"
        ).is_file()
        and (
            path / "msigdb"
            / f"h.all.v{locked_msigdb_release}.symbols.gmt"
        ).is_file()
    ),
    label="resources_biological con MSigDB y PAM50",
)
directional_signatures, _, _, signature_source = load_locked_signature_resources(
    results_root=RESULTS_ROOT,
    resource_dir=RESOURCE_DIR,
)
directional_signatures = {
    str(class_name): {str(gene).upper() for gene in members}
    for class_name, members in directional_signatures.items()
}
if set(CLASS_NAMES) - set(directional_signatures):
    raise RuntimeError("Faltan firmas direccionales de alguna clase.")

definitive_terms = PROTOCOL.get("biology_graph", {}).get(
    "definitive_locked_score_terms"
) or {
    "absolute_centered_class_delta": 1.0,
    "signed_context_agreement": 0.35,
    "cohort_support": 0.075,
}
CONTEXT_WEIGHT = float(definitive_terms["signed_context_agreement"])
SUPPORT_WEIGHT = float(definitive_terms["cohort_support"])
if not (
    math.isclose(CONTEXT_WEIGHT, 0.35, abs_tol=1e-12)
    and math.isclose(SUPPORT_WEIGHT, 0.075, abs_tol=1e-12)
):
    raise RuntimeError("Los pesos biológicos definitivos no son 1/.35/.075.")
biology_graph_cfg = PROTOCOL.get("biology_graph", {})
MODULE_POOL_MULTIPLIER = int(
    biology_graph_cfg.get("module_pool_multiplier", 6)
)
MODULE_TARGET_COMPONENTS = int(
    biology_graph_cfg.get("target_components", 12)
)
MODULE_MIN_EDGES = int(
    biology_graph_cfg.get("minimum_edges_per_seed_module", 3)
)


def dense_path(kappa, seed):
    return (
        ARTIFACTS_ROOT / ARTIFACT_BY_KAPPA[float(kappa)]
        / f"seed_{int(seed)}" / "transport_support_dense.npz"
    )


@lru_cache(maxsize=None)
def dense_metadata(path_text):
    path = Path(path_text)
    with np.load(path, allow_pickle=True) as dense:
        required = {
            "class_names", "structural_unit_id", "differential_logits",
            "context_score", "selection_frequency",
        }
        missing_dense = required - set(dense.files)
        if missing_dense:
            raise ValueError(f"{path}: faltan {sorted(missing_dense)}")
        class_names = tuple(map(str, dense["class_names"].tolist()))
        unit_id = np.asarray(dense["structural_unit_id"], dtype=np.int64)
    if unit_id.size != dense_edge_capacity:
        raise RuntimeError(
            f"{path}: capacidad={unit_id.size}; esperada={dense_edge_capacity}."
        )
    valid = candidate_ids[unit_id[candidate_ids] >= 0]
    units, counts = np.unique(unit_id[valid], return_counts=True)
    return {
        "classes": class_names,
        "n_valid_edges": int(len(valid)),
        "n_units": int(len(units)),
        "unit_widths": tuple(map(int, np.unique(counts))),
    }


dense_audit_rows = []
for kappa in KAPPA_GRID:
    for seed in SEEDS:
        path = dense_path(kappa, seed)
        meta = dense_metadata(str(path))
        if set(meta["classes"]) != set(CLASS_NAMES):
            raise RuntimeError(f"{path}: clases incompatibles.")
        if meta["unit_widths"] != (1,):
            raise RuntimeError(f"{path}: unidad estructural no unitaria.")
        if meta["n_units"] < K_BY_KAPPA[kappa]:
            raise RuntimeError(f"{path}: unidades insuficientes.")
        dense_audit_rows.append({
            "kappa": kappa, "K": K_BY_KAPPA[kappa], "seed": seed,
            "path": str(path), **meta,
        })
dense_audit = pd.DataFrame(dense_audit_rows)
dense_audit.to_csv(OUTPUT_DIR / "dense_artifact_audit.csv", index=False)
display(dense_audit[[
    "kappa", "K", "seed", "n_valid_edges", "n_units", "unit_widths",
]])


In [ ]:
def extract_trained_exact_k(dense_file, *, class_name, kappa, seed):
    dense_file = Path(dense_file)
    k_edges = int(K_BY_KAPPA[float(kappa)])
    with np.load(dense_file, allow_pickle=True) as dense:
        class_names = tuple(map(str, dense["class_names"].tolist()))
        class_id = class_names.index(str(class_name))
        unit_id = np.asarray(dense["structural_unit_id"], dtype=np.int64)
        valid = candidate_ids[unit_id[candidate_ids] >= 0]
        units, first_positions = np.unique(unit_id[valid], return_index=True)
        representative_ids = valid[first_positions]
        scores = combine_biology_scores(
            np.asarray(dense["differential_logits"], dtype=float),
            np.asarray(dense["context_score"], dtype=float),
            np.asarray(dense["selection_frequency"], dtype=float),
            context_weight=CONTEXT_WEIGHT,
            support_weight=SUPPORT_WEIGHT,
        )
        selected_local = module_aware_exact_k(
            full_edge_index[:, representative_ids],
            scores[class_id, representative_ids],
            k_edges=k_edges,
            pool_multiplier=MODULE_POOL_MULTIPLIER,
            target_components=MODULE_TARGET_COMPONENTS,
            min_component_edges=MODULE_MIN_EDGES,
        )
        selected_ids = representative_ids[selected_local]
        differential = np.asarray(
            dense["differential_logits"][class_id], dtype=float
        )
        context = np.asarray(dense["context_score"][class_id], dtype=float)
        support = np.asarray(
            dense["selection_frequency"][class_id], dtype=float
        )

    frame = candidate_lookup.loc[selected_ids].reset_index(drop=True).copy()
    frame["seed"] = int(seed)
    frame["class_name"] = str(class_name)
    frame["kappa"] = float(kappa)
    frame["K"] = k_edges
    frame["differential_logit"] = differential[selected_ids]
    frame["context_score"] = context[selected_ids]
    frame["selection_frequency"] = support[selected_ids]
    frame["biology_score"] = scores[class_id, selected_ids]
    frame = frame.sort_values(
        ["biology_score", "candidate_edge_id"],
        ascending=[False, True],
    ).reset_index(drop=True)
    frame.insert(0, "biology_rank", np.arange(1, len(frame) + 1))
    if len(frame) != k_edges or frame.candidate_edge_id.nunique() != k_edges:
        raise RuntimeError(
            f"κ={kappa}/{class_name}/seed={seed}: Exact-K roto."
        )
    return frame


def selection_stability(frames, *, universe_size):
    sets = [set(frame.candidate_edge_id.astype(int)) for frame in frames]
    if len(sets) < 2 or len({len(values) for values in sets}) != 1:
        raise RuntimeError("La estabilidad requiere ≥2 conjuntos de igual K.")
    k_edges = len(sets[0])
    pairwise = []
    for left, right in combinations(range(len(sets)), 2):
        intersection = len(sets[left] & sets[right])
        union = len(sets[left] | sets[right])
        expected = k_edges * k_edges / float(universe_size)
        pairwise.append({
            "jaccard": intersection / union if union else np.nan,
            "kuncheva": (
                (intersection - expected) / (k_edges - expected)
                if k_edges > expected else np.nan
            ),
        })
    matrix = np.zeros((len(sets), int(universe_size)), dtype=np.uint8)
    for row, values in enumerate(sets):
        ids = np.fromiter(values, dtype=np.int64)
        candidate_rows = candidate_id_to_row[ids]
        if np.any(candidate_rows < 0):
            raise RuntimeError("Una selección contiene aristas fuera del prior.")
        matrix[row, candidate_rows] = 1
    selected_fraction = float(matrix.sum(axis=1).mean() / universe_size)
    denominator = selected_fraction * (1 - selected_fraction)
    nogueira = 1.0 - float(matrix.var(axis=0, ddof=1).mean()) / denominator
    strict_ids = set.intersection(*sets)
    return {
        "mean_pairwise_jaccard": float(np.mean([x["jaccard"] for x in pairwise])),
        "minimum_pairwise_jaccard": float(np.min([x["jaccard"] for x in pairwise])),
        "mean_kuncheva": float(np.mean([x["kuncheva"] for x in pairwise])),
        "nogueira_stability": float(nogueira),
        "strict_edges": int(len(strict_ids)),
        "strict_fraction_of_K": float(len(strict_ids) / k_edges),
        "union_edges": int(len(set.union(*sets))),
    }


stratum_codes, _ = pd.factorize(candidates.null_stratum, sort=True)
stratum_codes = np.asarray(stratum_codes, dtype=np.int64)
n_strata = int(stratum_codes.max()) + 1
stratum_sizes = np.bincount(stratum_codes, minlength=n_strata).astype(float)
source_genes = candidates.source_gene.to_numpy(str)
target_genes = candidates.target_gene.to_numpy(str)
signature_hits = {}
signature_pool_means = {}
for signature_name, members in directional_signatures.items():
    source_hit = np.isin(source_genes, list(members))
    target_hit = np.isin(target_genes, list(members))
    for statistic, hits in {
        "at_least_one_endpoint": source_hit | target_hit,
        "both_endpoints": source_hit & target_hit,
    }.items():
        key = (signature_name, statistic)
        hits = hits.astype(float)
        signature_hits[key] = hits
        signature_pool_means[key] = (
            np.bincount(stratum_codes, weights=hits, minlength=n_strata)
            / stratum_sizes
        )


def matched_signature_enrichment(selected):
    ids = selected.candidate_edge_id.astype(int).to_numpy()
    if not len(ids):
        raise ValueError("No se evalúa un grafo vacío.")
    rows_selected = candidate_id_to_row[ids]
    if np.any(rows_selected < 0):
        raise RuntimeError("El grafo contiene aristas fuera del prior.")
    codes_selected = stratum_codes[rows_selected]
    rows = []
    for signature_name in CLASS_NAMES:
        for statistic in ("at_least_one_endpoint", "both_endpoints"):
            key = (signature_name, statistic)
            observed = float(signature_hits[key][rows_selected].mean())
            expected = float(
                signature_pool_means[key][codes_selected].mean()
            )
            fold = observed / expected if expected > 0 else np.nan
            rows.append({
                "signature": signature_name,
                "statistic": statistic,
                "observed_fraction": observed,
                "matched_expected_fraction": expected,
                "fold_over_matched_expectation": fold,
                "log2_fold": float(np.log2(fold)) if fold > 0 else -np.inf,
                "n_edges": len(ids),
            })
    return pd.DataFrame(rows)


def specificity_summary(selected):
    enrichment = matched_signature_enrichment(selected)
    rows = []
    for statistic, block in enrichment.groupby("statistic", sort=False):
        basal = float(
            block.loc[block.signature.eq(TARGET_CLASS), "log2_fold"].iloc[0]
        )
        alternatives = block.loc[
            ~block.signature.eq(TARGET_CLASS),
            ["signature", "log2_fold"],
        ].sort_values("log2_fold", ascending=False)
        best = alternatives.iloc[0]
        rows.append({
            "statistic": statistic,
            "basal_log2_fold": basal,
            "best_alternative_signature": str(best.signature),
            "best_alternative_log2_fold": float(best.log2_fold),
            "specificity_margin": basal - float(best.log2_fold),
        })
    return pd.DataFrame(rows), enrichment


In [ ]:
trained_graphs = {}
consensus_graphs = {}
structural_rows = []
topology_rows = []
biology_rows = []
specificity_rows = []

for kappa in KAPPA_GRID:
    k_edges = K_BY_KAPPA[kappa]
    print(f"[reconstrucción] κ={kappa:.3f} | K={k_edges}")
    kappa_dir = GRAPH_DIR / f"kappa_{kappa_tag(kappa)}"
    kappa_dir.mkdir(parents=True, exist_ok=True)
    for class_name in CLASS_NAMES:
        seed_graphs = {}
        class_dir = kappa_dir / safe_name(class_name)
        class_dir.mkdir(parents=True, exist_ok=True)
        for seed in SEEDS:
            frame = extract_trained_exact_k(
                dense_path(kappa, seed),
                class_name=class_name,
                kappa=kappa,
                seed=seed,
            )
            trained_graphs[(kappa, class_name, seed)] = frame
            seed_graphs[seed] = frame
            frame.to_csv(
                class_dir / f"seed_{seed}.csv.gz",
                index=False,
                compression="gzip",
            )

        consensus, strict, aggregate_summary = aggregate_seed_graphs(
            seed_graphs, k_edges=k_edges
        )
        consensus["kappa"] = kappa
        consensus["K"] = k_edges
        consensus["class_name"] = class_name
        strict["kappa"] = kappa
        strict["K"] = k_edges
        strict["class_name"] = class_name
        consensus_graphs[(kappa, class_name)] = consensus.copy()
        consensus.to_csv(
            class_dir / "consensus_exact_k.csv", index=False
        )
        strict.to_csv(class_dir / "strict_core_5of5.csv", index=False)

        structural_rows.append({
            "kappa": kappa,
            "K": k_edges,
            "class_name": class_name,
            **selection_stability(
                list(seed_graphs.values()), universe_size=N_CANDIDATE_EDGES
            ),
        })
        for graph_kind, graph in (("consensus", consensus), ("strict_5of5", strict)):
            topology_rows.append({
                "kappa": kappa,
                "K": k_edges,
                "class_name": class_name,
                "graph_kind": graph_kind,
                **topology_metrics(graph),
            })

        enrichment = matched_signature_enrichment(consensus)
        enrichment.insert(0, "graph_class", class_name)
        enrichment.insert(0, "K", k_edges)
        enrichment.insert(0, "kappa", kappa)
        biology_rows.append(enrichment)
        specificity, _ = specificity_summary(consensus)
        specificity.insert(0, "graph_class", class_name)
        specificity.insert(0, "K", k_edges)
        specificity.insert(0, "kappa", kappa)
        specificity_rows.append(specificity)

structural_stability = pd.DataFrame(structural_rows)
topology_summary = pd.DataFrame(topology_rows)
biology_enrichment = pd.concat(biology_rows, ignore_index=True)
signature_specificity = pd.concat(specificity_rows, ignore_index=True)

# Concordancia de cada red consenso con la especificación principal.
# Además de Jaccard se informa el coeficiente de solapamiento, menos
# penalizado por la diferencia mecánica de tamaño entre K distintos.
cross_k_rows = []
for class_name in CLASS_NAMES:
    reference = consensus_graphs[(PRIMARY_KAPPA, class_name)]
    reference_edges = set(zip(
        reference.source_gene.astype(str),
        reference.target_gene.astype(str),
    ))
    reference_genes = set(reference.source_gene) | set(reference.target_gene)
    for kappa in KAPPA_GRID:
        current = consensus_graphs[(kappa, class_name)]
        current_edges = set(zip(
            current.source_gene.astype(str),
            current.target_gene.astype(str),
        ))
        current_genes = set(current.source_gene) | set(current.target_gene)
        intersection = len(reference_edges & current_edges)
        union = len(reference_edges | current_edges)
        expected = (
            len(reference_edges) * len(current_edges)
            / float(N_CANDIDATE_EDGES)
        )
        maximum_above_chance = min(
            len(reference_edges), len(current_edges)
        ) - expected
        cross_k_rows.append({
            "kappa": kappa,
            "K": K_BY_KAPPA[kappa],
            "class_name": class_name,
            "reference_kappa": PRIMARY_KAPPA,
            "edge_intersection": intersection,
            "edge_jaccard_vs_primary": (
                intersection / union if union else np.nan
            ),
            "edge_overlap_coefficient_vs_primary": (
                intersection / min(len(reference_edges), len(current_edges))
                if reference_edges and current_edges else np.nan
            ),
            "chance_corrected_overlap_vs_primary": (
                (intersection - expected) / maximum_above_chance
                if maximum_above_chance > 0 else np.nan
            ),
            "gene_jaccard_vs_primary": (
                len(reference_genes & current_genes)
                / len(reference_genes | current_genes)
                if reference_genes or current_genes else np.nan
            ),
        })
cross_k_concordance = pd.DataFrame(cross_k_rows)
structural_stability.to_csv(
    OUTPUT_DIR / "kappa_structural_stability.csv", index=False
)
topology_summary.to_csv(
    OUTPUT_DIR / "kappa_topology_summary.csv", index=False
)
biology_enrichment.to_csv(
    OUTPUT_DIR / "kappa_signature_enrichment.csv", index=False
)
signature_specificity.to_csv(
    OUTPUT_DIR / "kappa_signature_specificity.csv", index=False
)
cross_k_concordance.to_csv(
    OUTPUT_DIR / "kappa_consensus_concordance_vs_primary.csv", index=False
)

display(structural_stability.loc[
    structural_stability.class_name.eq(TARGET_CLASS)
])
display(signature_specificity.loc[
    signature_specificity.graph_class.eq(TARGET_CLASS)
    & signature_specificity.statistic.eq("at_least_one_endpoint")
])
display(cross_k_concordance.loc[
    cross_k_concordance.class_name.eq(TARGET_CLASS)
])


## 6. Dependencia de GNN y message passing

Estas perturbaciones se aplican a los modelos ya entrenados y únicamente
en desarrollo-validación. Permiten comprobar que el uso funcional de la
rama gráfica no desaparece al cambiar κ; no sustituyen los controles
topológicos reentrenados del notebook 04.


In [ ]:
X_FULL = np.asarray(runtime["X_scaled"], dtype=np.float32)
X_H = np.asarray(runtime["X_h"], dtype=np.float32)
Y = np.asarray(runtime["y"], dtype=np.int64)
VAL_IDX = np.asarray(runtime["val_idx"], dtype=np.int64)
LABEL_MAP = dict(runtime["label_map"])


@contextmanager
def zero_message_passing(model):
    core = getattr(model, "gnn", model)
    names = ["edge_weight_norm_base"] + [
        f"fixed_class_edge_weight_norm_{class_id}"
        for class_id in range(int(getattr(core, "n_classes", 0)))
    ]
    tensors = [getattr(core, name) for name in names if hasattr(core, name)]
    if not tensors:
        raise RuntimeError("No se encontraron pesos fijos de mensajes.")
    backups = [tensor.detach().clone() for tensor in tensors]
    with torch.no_grad():
        for tensor in tensors:
            tensor.zero_()
    try:
        yield
    finally:
        with torch.no_grad():
            for tensor, backup in zip(tensors, backups):
                tensor.copy_(backup)


def validation_branch_audit(kappa, seed):
    run_root = (
        ARTIFACTS_ROOT / ARTIFACT_BY_KAPPA[float(kappa)]
        / f"seed_{int(seed)}"
    )
    bundle = load_bundle(str(run_root))
    model = build_pruned_model_from_bundle(bundle, device=CFG.DEVICE)
    model.eval()
    nodes = np.asarray(bundle["graph"]["nodes_used_full"], dtype=np.int64)
    flags = bundle.get("model", {}).get("model_flags", {})
    X_comp = (
        np.hstack([X_FULL[:, nodes], X_FULL]).astype(np.float32)
        if bool(flags.get("tab_full_gene_universe", False))
        else X_FULL[:, nodes]
    )
    loader = torch.utils.data.DataLoader(
        ExpressionDataset(X_comp, X_H, Y, VAL_IDX),
        batch_size=int(CFG.BATCH_SIZE),
        shuffle=False,
        num_workers=0,
    )
    variants = (
        ("complete", None),
        ("without_tabular_branch", "tab"),
        ("without_GNN", "gnn"),
        ("without_message_passing", "messages"),
    )
    rows = []
    for variant, disabled in variants:
        module = (
            model.tab_trunk if disabled == "tab"
            else model.gnn.trunk if disabled == "gnn"
            else None
        )
        hook = (
            module.register_forward_hook(
                lambda _module, _inputs, output: torch.zeros_like(output)
            ) if module is not None else None
        )
        context = (
            zero_message_passing(model)
            if disabled == "messages" else nullcontext()
        )
        try:
            with context, torch.no_grad():
                probability, truth = predict_proba_xh_mode(
                    model, None, loader, CFG.DEVICE, xh_mode="orig"
                )
            rows.append({
                "kappa": float(kappa),
                "K": int(K_BY_KAPPA[float(kappa)]),
                "seed": int(seed),
                "variant": variant,
                **compute_metrics_full(truth, probability, LABEL_MAP),
            })
        finally:
            if hook is not None:
                hook.remove()
    del model, loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rows


branch_rows = []
branch_file = OUTPUT_DIR / "kappa_branch_reliance_validation.csv"
if RUN_BRANCH_AUDIT:
    for kappa in KAPPA_GRID:
        for seed in SEEDS:
            print(f"[ramas] κ={kappa:.3f}, seed={seed}", flush=True)
            branch_rows.extend(validation_branch_audit(kappa, seed))
            pd.DataFrame(branch_rows).to_csv(branch_file, index=False)
elif branch_file.is_file():
    branch_rows = pd.read_csv(branch_file).to_dict(orient="records")
else:
    raise FileNotFoundError(
        "RUN_BRANCH_AUDIT=0 pero no existe el resultado previo."
    )

branch_results = pd.DataFrame(branch_rows)
expected_branch_rows = len(KAPPA_GRID) * len(SEEDS) * 4
if len(branch_results) != expected_branch_rows:
    raise RuntimeError(
        f"Auditoría de ramas incompleta: {len(branch_results)}; "
        f"esperadas={expected_branch_rows}."
    )
base = branch_results.loc[
    branch_results.variant.eq("complete"),
    ["kappa", "seed", "macro_f1", "auc_macro_ovr"],
].rename(columns={
    "macro_f1": "base_macro_f1", "auc_macro_ovr": "base_auc",
})
branch_drops = branch_results.merge(base, on=["kappa", "seed"])
branch_drops["macro_f1_drop"] = (
    branch_drops.base_macro_f1 - branch_drops.macro_f1
)
branch_drops["auc_drop"] = branch_drops.base_auc - branch_drops.auc_macro_ovr
branch_summary = (
    branch_drops.groupby(["kappa", "K", "variant"])
    .agg(
        macro_f1_drop_mean=("macro_f1_drop", "mean"),
        macro_f1_drop_std=("macro_f1_drop", "std"),
        auc_drop_mean=("auc_drop", "mean"),
        auc_drop_std=("auc_drop", "std"),
    )
    .reset_index()
)
branch_drops.to_csv(branch_file, index=False)
branch_summary.to_csv(
    OUTPUT_DIR / "kappa_branch_reliance_summary.csv", index=False
)
display(branch_summary)


## 7. Tabla integrada, figura suplementaria y criterios predefinidos


In [ ]:
# Una fila por κ con todos los criterios principales.
pred = predictive_summary.copy()
pred = pred.rename(columns={
    "K_nominal": "K", "K_nominal_": "K",
})
paired_f1 = predictive_paired.loc[
    predictive_paired.metric.eq("macro_f1"),
    [
        "kappa", "K", "mean_delta_vs_primary",
        "paired_seed_ci95_lower", "paired_seed_ci95_upper",
        "macro_f1_noninferior_diagnostic",
    ],
]
basal_stability = structural_stability.loc[
    structural_stability.class_name.eq(TARGET_CLASS)
].drop(columns="class_name")
basal_topology = topology_summary.loc[
    topology_summary.class_name.eq(TARGET_CLASS)
    & topology_summary.graph_kind.eq("consensus")
].drop(columns=["class_name", "graph_kind"])
basal_enrichment = biology_enrichment.loc[
    biology_enrichment.graph_class.eq(TARGET_CLASS)
    & biology_enrichment.signature.eq(TARGET_CLASS),
    [
        "kappa", "K", "statistic",
        "fold_over_matched_expectation", "observed_fraction",
        "matched_expected_fraction",
    ],
].pivot(index=["kappa", "K"], columns="statistic").reset_index()
basal_enrichment.columns = [
    "_".join(str(value) for value in column if str(value))
    if isinstance(column, tuple) else str(column)
    for column in basal_enrichment.columns
]
basal_enrichment = basal_enrichment.rename(columns={
    "kappa_": "kappa", "K_": "K",
})
basal_specificity = signature_specificity.loc[
    signature_specificity.graph_class.eq(TARGET_CLASS)
    & signature_specificity.statistic.eq("at_least_one_endpoint"),
    [
        "kappa", "K", "specificity_margin",
        "best_alternative_signature", "basal_log2_fold",
    ],
]
basal_cross_k = cross_k_concordance.loc[
    cross_k_concordance.class_name.eq(TARGET_CLASS)
].drop(columns=["class_name", "reference_kappa"])
relevant_branches = branch_summary.loc[
    branch_summary.variant.isin(["without_GNN", "without_message_passing"]),
    ["kappa", "K", "variant", "macro_f1_drop_mean", "auc_drop_mean"],
].pivot(index=["kappa", "K"], columns="variant").reset_index()
relevant_branches.columns = [
    "_".join(str(value) for value in column if str(value))
    if isinstance(column, tuple) else str(column)
    for column in relevant_branches.columns
]
relevant_branches = relevant_branches.rename(columns={
    "kappa_": "kappa", "K_": "K",
})

summary = pred.merge(paired_f1, on=["kappa", "K"], how="left")
for table in (
    basal_stability,
    basal_topology,
    basal_enrichment,
    basal_specificity,
    basal_cross_k,
    relevant_branches,
):
    summary = summary.merge(table, on=["kappa", "K"], how="left")

fold_one_col = "fold_over_matched_expectation_at_least_one_endpoint"
fold_both_col = "fold_over_matched_expectation_both_endpoints"
summary["basal_recovery_above_matched_expectation"] = (
    summary[fold_one_col].gt(1.0) & summary[fold_both_col].gt(1.0)
)
summary["basal_specificity_positive"] = summary.specificity_margin.gt(0.0)
summary["graph_reliance_positive"] = (
    summary["macro_f1_drop_mean_without_GNN"].gt(0.0)
    & summary["macro_f1_drop_mean_without_message_passing"].gt(0.0)
)
summary.to_csv(
    OUTPUT_DIR / "Supplementary_Table_Kappa_retraining_sensitivity.csv",
    index=False,
)
display(summary.T)

palette = {
    "primary": "#d73027",
    "alternative": "#2878b5",
    "secondary": "#7a3e9d",
}
fig, axes = plt.subplots(2, 3, figsize=(15.2, 8.5), constrained_layout=True)
x = summary.kappa.to_numpy(float)

axes[0, 0].errorbar(
    x, summary.macro_f1_mean, yerr=summary.macro_f1_std,
    marker="o", linewidth=2, capsize=3, color=palette["alternative"],
)
axes[0, 0].set(title="A  Rendimiento de desarrollo", ylabel="Macro-F1 (media ± DE)")

axes[0, 1].plot(
    x, summary.macro_f1_drop_mean_without_GNN,
    marker="o", linewidth=2, label="Sin GNN",
)
axes[0, 1].plot(
    x, summary.macro_f1_drop_mean_without_message_passing,
    marker="s", linewidth=2, label="Sin message passing",
)
axes[0, 1].axhline(0, color="#777777", linestyle="--", linewidth=1)
axes[0, 1].set(title="B  Dependencia de la rama gráfica", ylabel="Caída de macro-F1")
axes[0, 1].legend(frameon=False)

axes[0, 2].plot(
    x, summary.nogueira_stability,
    marker="o", linewidth=2, label="Nogueira",
)
axes[0, 2].plot(
    x, summary.mean_pairwise_jaccard,
    marker="s", linewidth=2, label="Jaccard",
)
axes[0, 2].plot(
    x, summary.edge_overlap_coefficient_vs_primary,
    marker="^", linewidth=2, label="Solapamiento con κ=0.010",
)
axes[0, 2].set(
    title="C  Estabilidad y concordancia Basal-like",
    ylabel="Índice",
)
axes[0, 2].legend(frameon=False)

axes[1, 0].plot(
    x, summary.strict_fraction_of_K,
    marker="o", linewidth=2, color=palette["secondary"],
)
axes[1, 0].set(title="D  Núcleo Basal-like 5/5", ylabel="|núcleo| / K")

axes[1, 1].plot(
    x, summary[fold_one_col], marker="o", linewidth=2,
    label="≥1 extremo",
)
axes[1, 1].plot(
    x, summary[fold_both_col], marker="s", linewidth=2,
    label="Ambos extremos",
)
axes[1, 1].axhline(1, color="#777777", linestyle="--", linewidth=1)
axes[1, 1].set(title="E  Recuperación Basal-like", ylabel="Fold / expectativa emparejada")
axes[1, 1].legend(frameon=False)

axes[1, 2].plot(
    x, summary.specificity_margin,
    marker="o", linewidth=2, color=palette["primary"],
)
axes[1, 2].axhline(0, color="#777777", linestyle="--", linewidth=1)
axes[1, 2].set(title="F  Especificidad Basal-like", ylabel="Margen log2-fold")

for axis in axes.flat:
    axis.axvline(
        PRIMARY_KAPPA, color=palette["primary"],
        linestyle=":", linewidth=1.3,
    )
    axis.set_xlabel("κ")
    axis.set_xticks(KAPPA_GRID)
    axis.grid(axis="y", color="#dddddd", linewidth=0.7)
fig.suptitle(
    "Sensibilidad al objetivo de sparsidad con reentrenamiento completo",
    fontsize=15,
    fontweight="bold",
)
for extension in ("png", "pdf", "svg"):
    fig.savefig(
        FIGURE_DIR / f"Supplementary_Kappa_retraining_sensitivity.{extension}",
        dpi=350 if extension == "png" else None,
        bbox_inches="tight",
    )
plt.show()


## 8. Informe automático y manifiesto


In [ ]:
alternatives = summary.loc[~summary.kappa.eq(PRIMARY_KAPPA)].copy()
predictive_noninferior = bool(
    alternatives.macro_f1_noninferior_diagnostic.astype(bool).all()
)
biological_robust = bool(
    summary.basal_recovery_above_matched_expectation.all()
    and summary.basal_specificity_positive.all()
)
graph_reliance_robust = bool(summary.graph_reliance_positive.all())

report_text = dedent(f'''\
# Sensibilidad de κ con reentrenamiento

Se reentrenó el flujo completo en cinco objetivos de sparsidad (κ={min(KAPPA_GRID):.3f}–{max(KAPPA_GRID):.3f}; K={min(K_BY_KAPPA.values())}–{max(K_BY_KAPPA.values())} aristas por clase) manteniendo constantes semillas, particiones, presupuesto de optimización, pérdidas, arquitectura y métrica de checkpoint. El modelo principal κ={PRIMARY_KAPPA:.3f} se reutilizó y los cuatro valores alternativos aportaron veinte entrenamientos nuevos. Todas las métricas proceden de desarrollo-validación.

El macro-F1 medio varió entre {summary.macro_f1_mean.min():.3f} y {summary.macro_f1_mean.max():.3f}. El diagnóstico pareado de no inferioridad con margen −{NONINFERIORITY_MARGIN:.2f} fue {'compatible en las cuatro densidades alternativas' if predictive_noninferior else 'no compatible en al menos una densidad alternativa'}.

En las redes Basal-like, el Jaccard medio entre semillas fue {summary.mean_pairwise_jaccard.min():.3f}–{summary.mean_pairwise_jaccard.max():.3f}, la estabilidad de Nogueira {summary.nogueira_stability.min():.3f}–{summary.nogueira_stability.max():.3f} y la fracción del núcleo 5/5 {summary.strict_fraction_of_K.min():.3f}–{summary.strict_fraction_of_K.max():.3f}. El coeficiente de solapamiento de los consensos alternativos con κ={PRIMARY_KAPPA:.3f} fue {alternatives.edge_overlap_coefficient_vs_primary.min():.3f}–{alternatives.edge_overlap_coefficient_vs_primary.max():.3f}. La recuperación Basal-like permaneció en {summary[fold_one_col].min():.2f}–{summary[fold_one_col].max():.2f} veces la expectativa emparejada para al menos un extremo y {summary[fold_both_col].min():.2f}–{summary[fold_both_col].max():.2f} para ambos extremos. El margen de especificidad fue {summary.specificity_margin.min():.3f}–{summary.specificity_margin.max():.3f} log2-fold.

Conclusión del análisis: recuperación biológica robusta={'sí' if biological_robust else 'no'}; dependencia positiva de GNN y message passing en toda la rejilla={'sí' if graph_reliance_robust else 'no'}; diagnóstico predictivo pareado compatible con el margen predefinido={'sí' if predictive_noninferior else 'no'}. κ={PRIMARY_KAPPA:.3f} permanece como especificación principal y estos resultados se interpretan exclusivamente como sensibilidad, no como una nueva selección de hiperparámetros.
''').strip() + "\n"
(OUTPUT_DIR / "manuscript_ready_kappa_retraining.md").write_text(
    report_text, encoding="utf-8"
)

output_files = [
    OUTPUT_DIR / "kappa_configuration_isolation_audit.csv",
    OUTPUT_DIR / "kappa_development_results.csv",
    OUTPUT_DIR / "kappa_bundle_integrity_audit.csv",
    OUTPUT_DIR / "kappa_predictive_summary.csv",
    OUTPUT_DIR / "kappa_predictive_paired_vs_primary.csv",
    OUTPUT_DIR / "kappa_structural_stability.csv",
    OUTPUT_DIR / "kappa_topology_summary.csv",
    OUTPUT_DIR / "kappa_signature_enrichment.csv",
    OUTPUT_DIR / "kappa_signature_specificity.csv",
    OUTPUT_DIR / "kappa_consensus_concordance_vs_primary.csv",
    OUTPUT_DIR / "kappa_branch_reliance_validation.csv",
    OUTPUT_DIR / "kappa_branch_reliance_summary.csv",
    OUTPUT_DIR / "Supplementary_Table_Kappa_retraining_sensitivity.csv",
    OUTPUT_DIR / "manuscript_ready_kappa_retraining.md",
]
manifest = {
    "status": "complete",
    "analysis": "kappa_training_target_sensitivity",
    "primary_kappa_retained": PRIMARY_KAPPA,
    "kappa_grid": list(map(float, KAPPA_GRID)),
    "K_grid": [int(K_BY_KAPPA[kappa]) for kappa in KAPPA_GRID],
    "seeds": list(map(int, SEEDS)),
    "new_complete_training_runs": (len(KAPPA_GRID) - 1) * len(SEEDS),
    "phase1_and_phase2_retrained": True,
    "external_opened": False,
    "evaluation_split": "development_validation",
    "only_authorized_model_field_changed": "KEEP_MIN",
    "predictive_noninferior_diagnostic": predictive_noninferior,
    "biological_recovery_robust": biological_robust,
    "graph_reliance_robust": graph_reliance_robust,
    "input_hashes": {
        "protocol": sha256_file(PROTOCOL_FILE),
        "candidate_prior": sha256_file(CANDIDATE_FILE),
        "signature_manifest": sha256_file(SIGNATURE_MANIFEST_FILE),
        "implementation": IMPLEMENTATION_SHA256,
        "primary_model_config": BASE_FINGERPRINT,
    },
    "output_hashes": {
        path.name: sha256_file(path)
        for path in output_files if path.is_file()
    },
}
MANIFEST_FILE = OUTPUT_DIR / "KAPPA_RETRAINING_MANIFEST.json"
MANIFEST_FILE.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

display(Markdown(report_text))
print("\nResultados:", OUTPUT_DIR)
print("Manifiesto:", MANIFEST_FILE)
print("Figura:", FIGURE_DIR / "Supplementary_Kappa_retraining_sensitivity.pdf")


## Referencias metodológicas

- Liu H, Roeder K, Wasserman L. *Stability Approach to Regularization
  Selection (StARS) for High Dimensional Graphical Models*. NeurIPS.
  2010. [Texto completo](https://pmc.ncbi.nlm.nih.gov/articles/PMC4138724/).
  Sustenta el principio de estudiar conjuntamente sparsidad y
  reproducibilidad a lo largo de una trayectoria de regularización.
- Nogueira S, Sechidis K, Brown G. *On the Stability of Feature
  Selection Algorithms*. Journal of Machine Learning Research.
  2018;18(174):1–54.
  [Artículo](https://jmlr.org/papers/v18/17-514.html). Fundamenta la
  estimación de estabilidad corregida por azar.
- Kuncheva LI. *A stability index for feature selection*. Proceedings
  of the 25th IASTED International Multi-Conference: Artificial
  Intelligence and Applications. 2007:390–395.
  [Registro](https://research.bangor.ac.uk/en/publications/a-stability-index-for-feature-selection/).
- Cawley GC, Talbot NLC. *On Over-fitting in Model Selection and
  Subsequent Selection Bias in Performance Evaluation*. Journal of
  Machine Learning Research. 2010;11:2079–2107.
  [Artículo](https://www.jmlr.org/papers/v11/cawley10a.html). Motiva
  mantener κ=0.010 como especificación principal y no seleccionar
  retrospectivamente la densidad con resultados más favorables.

El uso de StARS es conceptual: este cuaderno no implementa StARS ni
presupone un modelo gráfico gaussiano.
